# CI/CD + Observability: Pipeline Metrics and Health Dashboards

This notebook explores how observability practices apply to CI/CD pipelines themselves—tracking DORA metrics, simulating pipeline events, and understanding health dashboard patterns. It combines CI/CD Pipeline Concepts with Observability & Monitoring Concepts.

## Why observability for CI/CD

Observability is typically discussed for production services, but CI/CD pipelines generate the same kinds of signals: event frequency, latency, and failure rates. Treating pipeline telemetry as a first-class data stream lets you detect regressions in developer velocity before they become team-wide pain. The DORA metrics—Deployment Frequency, Lead Time for Changes, Change Failure Rate, and Mean Time to Recover—provide a standardized view of pipeline health.

In [ ]:
"""
DORA metric definitions with elite performance targets.
These four metrics are the standard observability contract for CI/CD pipelines.
"""

DORA_METRICS = {
    "deployment_frequency": {
        "description": "How often a team deploys to production",
        "elite": "on-demand deploys",
        "unit": "deploys/week",
    },
    "lead_time": {
        "description": "Time from code commit to production running",
        "elite": "less than 1 day",
        "unit": "hours",
    },
    "change_failure_rate": {
        "description": "Percentage of deployments that fail and require remediation",
        "elite": "less than 15%",
        "unit": "percent",
    },
    "mttr": {
        "description": "Time to restore service after a failure",
        "elite": "less than 1 hour",
        "unit": "hours",
    },
}

for name, meta in DORA_METRICS.items():
    print(f"{name}: {meta['description']} (elite: {meta['elite']})")

## Simulating pipeline events

A CI/CD pipeline is a series of timestamped events: push received, build started, build completed, tests run, deploy started, deploy completed. Logging these events is the observability foundation—without captured timestamps, you cannot compute lead time or detect build-duration regressions.

The script below simulates pipeline event data and computes the four DORA metrics from raw run records.

In [ ]:
from datetime import datetime, timedelta
from collections import Counter
import random

random.seed(42)


def simulate_runs(n=20, start_date=None):
    """Return a list of pipeline run records for metric computation."""
    if start_date is None:
        start_date = datetime(2026, 7, 1)
    runs = []
    for i in range(n):
        commit_time = start_date + timedelta(
            days=i * 2 + random.randint(0, 1),
            hours=random.randint(6, 18),
        )
        deploy_time = commit_time + timedelta(
            hours=random.randint(1, 12) if random.random() > 0.15 else random.randint(24, 48)
        )
        failed = random.random() < 0.18
        runs.append({
            "id": f"run-{i+1:03d}",
            "commit_time": commit_time,
            "deploy_time": deploy_time if not failed else None,
            "failure": failed,
            "duration_min": (deploy_time - commit_time).total_seconds() / 60
                            if not failed else None,
        })
    return runs


runs = simulate_runs(20)

# Deployment frequency: count non-failed, deployed runs per week
first_week_start = runs[0]["commit_time"]
last_week_end = runs[-1]["commit_time"] + timedelta(days=7)
span_days = (last_week_end - first_week_start).days
weeks = max(span_days / 7, 1)
deploys = [r for r in runs if not r["failure"]]
deployment_frequency = len(deploys) / weeks

# Lead time: mean duration of successful runs in hours
successful = [r for r in runs if not r["failure"]]
mean_lead_time_h = (
    sum(r["duration_min"] for r in successful) / 60 / len(successful)
    if successful else 0
)

# Change failure rate: percentage of deployed runs that fail
change_failure_rate = (sum(1 for r in runs if r["failure"]) / len(runs)) * 100

# MTTR: not available in synthetic data; represented as mean duration of failed runs
failed_runs = [r for r in runs if r["failure"]]
# For this simulation, MTTR is approximated by the time from commit to detection
mttr_h = (
    sum(r["duration_min"] for r in failed_runs) / 60 / len(failed_runs)
    if failed_runs else 0
)

print(f"Deployment frequency: {deployment_frequency:.1f} deploys/week")
print(f"Mean lead time: {mean_lead_time_h:.1f} hours")
print(f"Change failure rate: {change_failure_rate:.1f}%")
print(f"Mean time to recover (approx): {mttr_h:.1f} hours")

## Health dashboards

A pipeline health dashboard translates raw events into actionable visuals. From an observability standpoint, the standard pattern is collect → store → display → alert. Applied to CI/CD, this means capturing pipeline state transitions, persisting them in a time-series store, rendering duration and success-rate trends, and alerting when thresholds breach.

Typical dashboard panels for CI/CD:
- Deployment frequency trend (line chart over weeks)
- Build duration p50 / p95 (histogram) to catch caching regressions
- Test flake rate (failed tests that pass on rerun)
- Deploy success by environment (staging vs production)
- Queue time (time from commit to build start)

This pattern combines observability (metrics from events) with CI/CD (the event source).

In [ ]:
"""
Compute a pipeline health score from multiple signals.
Higher is healthier (0-100 scale).
"""


def compute_health_score(runs, max_queue_min=30):
    """Aggregate signals into a 0-100 health score."""
    if not runs:
        return 0

    # 1. Success rate (40% weight)
    success_rate = (sum(1 for r in runs if not r["failure"]) / len(runs)) * 100
    score_success = success_rate * 0.4

    # 2. Duration efficiency (25% weight): penalize runs taking longer than 4 hours
    successful = [r for r in runs if not r["failure"]]
    avg_dur_min = (
        sum(r["duration_min"] for r in successful) / len(successful)
        if successful else 240
    )
    dur_score = max(0, 25 - (avg_dur_min / 240) * 25)

    # 3. Queue time (15% weight): add a synthetic queue_min to each run for demo
    queue_times = [random.randint(1, max_queue_min) for _ in runs]
    avg_queue = sum(queue_times) / len(queue_times)
    queue_score = max(0, 15 - (avg_queue / max_queue_min) * 15)

    # 4. Consistency (20% weight): lower std dev of duration is better
    if len(successful) > 1:
        dur_vals = [r["duration_min"] for r in successful]
        mean_dur = sum(dur_vals) / len(dur_vals)
        var = sum((x - mean_dur) ** 2 for x in dur_vals) / (len(dur_vals) - 1)
        std = var ** 0.5
        consistency_score = max(0, 20 - (std / 120) * 20)
    else:
        consistency_score = 10

    total = score_success + dur_score + queue_score + consistency_score
    return round(min(total, 100), 1)


runs = simulate_runs(25)
score = compute_health_score(runs)
print(f"Pipeline health score: {score}/100")
print("Signals: success_rate, avg_duration, queue_time, duration_consistency")

## Anti-patterns in pipeline observability

Common pitfalls undermine pipeline metrics even when the data is available:

- **45-minute pipeline** — developers stop trusting it and skip running it. Keep feedback loops under 5 minutes for lint and unit tests; reserve longer stages for nightly windows.
- **No artifact versioning** — staging and production run different builds, which creates environment drift and makes failures unreproducible.
- **Secrets in pipeline UI** — storing credentials in the CI/CD web UI instead of a secret manager like HashiCorp Vault or AWS Secrets Manager exposes them to anyone with pipeline read access.

## Artifact immutability

"Build once, deploy everywhere" is the pattern that ties CI/CD to container orchestration. A single immutable container image passes through dev, test, staging, and production unchanged. Environment differences are injected at runtime via environment variables, never baked into the artifact.

Immutability means every environment runs exactly the same bits. If staging passes and prod fails, the delta is configuration or data—not the artifact—so debugging is tractable.

In [ ]:
"""
Simulate artifact immutability checks across environments.
The same image tag must appear in every stage without modification.
"""


def validate_artifact_propagation(environments, image_tag):
    """Return a dict of {env: tag} for environment checks."""
    return {env: image_tag for env in environments}


environments = ["dev", "test", "staging", "prod"]
image_tag = "python-api:2026-07-30-abc123"
manifest = validate_artifact_propagation(environments, image_tag)

print("Artifact manifest:")
for env, tag in manifest.items():
    status = "OK" if tag == image_tag else "DRIFT"
    print(f"  {env}: {tag}  [{status}]")

# Verify no drift
all_same = all(tag == image_tag for tag in manifest.values())
print(f"\nAll environments use identical artifact: {all_same}")

## Conclusion

This notebook showed three integrations between CI/CD and observability:

1. **Metrics** — DORA metrics standardize pipeline health using Deployment Frequency, Lead Time, Change Failure Rate, and MTTR.
2. **Dashboards** — A health dashboard surfaces build duration, queue time, test flake rate, and deploy success by environment using the collect → store → display → alert pattern.
3. **Artifacts** — Immutable image propagation ensures every environment runs the same artifact, turning environment-level failures into configuration problems rather than binary-diff debugging.

These patterns map directly to the tooling that follows this concept—GitHub Actions and Jenkins produce the events, Prometheus stores the metrics, and Grafana renders the dashboard panels.